In [1]:
import ccxt
import pandas as pd
import ta
import numpy as np
import vectorbt as vbt
import warnings
warnings.filterwarnings("ignore")
import time

In [ ]:
# ccxt 라이브러리를 사용하여 업비트와 빗썸 거래소에서 XRP/KRW의 1분봉 데이터를 가져옵니다.
# 1회 200개의 데이터만 가져올 수 있으며, KST가 적용되어 있지 않습니다.
exchange_upb = ccxt.upbit()
exchange_bth = ccxt.bithumb()
data_upb = exchange_upb.fetch_ohlcv('XRP/KRW', '1m', limit=365)
data_bth = exchange_bth.fetch_ohlcv('XRP/KRW', '1m', limit=365)
df_upb = pd.DataFrame(data_upb, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
df_bth = pd.DataFrame(data_bth, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
df_upb['timestamp'] = pd.to_datetime(df_upb['timestamp'], unit='ms')
df_bth['timestamp'] = pd.to_datetime(df_bth['timestamp'], unit='ms')
#df_upb.set_index('timestamp', inplace=True)
#df_bth.set_index('timestamp', inplace=True)


In [ ]:
df_upb
#df_bth


In [ ]:
# vbt 라이브러리를 사용하여 업비트 거래소에서 XRP/KRW의 1분봉 데이터를 가져옵니다.
# => 44,640 개이어야 하나, len()이 그것에 모자람(왜 띠엄띠엄 받는지 모르겠음). 아울러 시간이 KST 기준이 아님
data_upb = vbt.CCXTData.download(
    exchange='upbit', 
    symbols='XRP/KRW', 
    timeframe='1m', 
    start='2026-10-01 00:00:00 KST', 
    end='2026-10-31 23:59:59 KST',
    retries=5,
    delay=100)

df = data_upb.get()
print(df.head())
print(df.tail())
print(len(df))

0it [00:00, ?it/s]

                             Open    High     Low   Close        Volume
Open time                                                              
2026-09-30 20:00:00+00:00  2027.0  2032.0  2026.0  2027.0  66684.780295
2026-09-30 20:01:00+00:00  2029.0  2030.0  2025.0  2026.0  97397.567819
2026-09-30 20:02:00+00:00  2028.0  2028.0  2026.0  2026.0  16089.552272
2026-09-30 20:03:00+00:00  2027.0  2028.0  2026.0  2028.0  14537.418609
2026-09-30 20:04:00+00:00  2028.0  2028.0  2025.0  2027.0  80450.772518
                             Open    High     Low   Close        Volume
Open time                                                              
2026-10-31 11:07:00+00:00  1682.0  1682.0  1681.0  1681.0   3108.268379
2026-10-31 11:08:00+00:00  1682.0  1682.0  1680.0  1680.0  34357.430563
2026-10-31 11:09:00+00:00  1681.0  1681.0  1680.0  1681.0  27389.719645
2026-10-31 11:10:00+00:00  1681.0  1681.0  1680.0  1680.0  10000.459639
2026-10-31 11:11:00+00:00  1681.0  1681.0  1680.0  1680.0   6267

In [11]:
a=(int(pd.Timestamp("2026-10-01 00:00:00", tz='Asia/Seoul').timestamp() * 1000)/60000)
b=(int(pd.Timestamp("2026-10-31 23:59:59", tz='Asia/Seoul').timestamp() * 1000)/60000)
#print(a, b)
#print(b-a)
c=ccxt.upbit().iso8601('2026-10-01 00:00:00')
print(type(c))

<class 'NoneType'>


In [2]:
# upbit/bithumb 거래소에서 특정 기간 동안의 XRP/KRW의 1분봉 데이터를 가져오는 함수 정의
def fetch_1m_ohlcv(exchange='upbit', symbol='XRP/KRW', timeframe='1m', start_str='2026-11-01 00:00:00', end_str='2026-11-07 14:59:59'):
    exchange = ccxt.upbit({'enableRateLimit': True}) if exchange=='upbit' else ccxt.bithumb({'enableRateLimit': True})
    
    start_ts = int(pd.Timestamp(start_str, tz='Asia/Seoul').timestamp() * 1000)
    end_ts = int(pd.Timestamp(end_str, tz='Asia/Seoul').timestamp() * 1000)
    
    timeframe_minutes = 1
    timeframe_ms = timeframe_minutes * 60 * 1000
    candle_count = ((end_ts - start_ts) // timeframe_ms) + 1
    
    all_candles = []
    to_ts = end_ts
    
    print(f"[{symbol}] 분봉 데이터 수집 시작 from {start_str} to {end_str}. candle_count: {candle_count} ea.")    

    while True:
        params = {'to': exchange.iso8601(to_ts)}
        try:
            candles = exchange.fetch_ohlcv(symbol=symbol, timeframe=timeframe, limit=200, params=params)
        except Exception as e:
            print(f"Error fetching data: {e}. Retrying in 2 seconds...")
            time.sleep(2)
            continue
        
        if not candles:
            break
        
        all_candles = candles + all_candles  # Prepend new candles to the list
        oldest_ts = candles[0][0]  # 첫번째 캔들의 타임스탬프를 가져옵니다.
        
        if oldest_ts <= start_ts:
            break
        
        to_ts = oldest_ts - 1  # Move to the next batch of candles
        time.sleep(0.2)  # Sleep to respect rate limits
        
    seen = set()
    deduped = []
    for c in all_candles:
        if c[0] not in seen and start_ts <= c[0] <= end_ts:
            seen.add(c[0])
            deduped.append(c)
            
    deduped.sort(key=lambda x: x[0])  # Sort by timestamp
    
    df = pd.DataFrame(deduped, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
    df['datetime'] = pd.to_datetime(df['timestamp'], unit='ms').dt.tz_localize('UTC').dt.tz_convert('Asia/Seoul')
    df.set_index('datetime', inplace=True)
    df.drop(columns=['timestamp'], inplace=True)
    return df


In [3]:
df_upb_1xrp = fetch_1m_ohlcv(exchange='upbit', 
                            symbol='XRP/KRW', 
                            timeframe='1m',
                            start_str='2026-11-01 00:00:00',
                            end_str='2026-11-08 14:59:59')

print(f"Fetched {len(df_upb_1xrp)} rows of data for XRP/KRW from Upbit.")
print(df_upb_1xrp.head())
print(df_upb_1xrp.tail())


[XRP/KRW] 분봉 데이터 수집 시작 from 2026-11-01 00:00:00 to 2026-11-08 14:59:59. candle_count: 10980 ea.
Fetched 10980 rows of data for XRP/KRW from Upbit.
                             open    high     low   close        volume
datetime                                                               
2026-11-01 00:00:00+09:00  1681.0  1682.0  1681.0  1681.0   8965.413178
2026-11-01 00:01:00+09:00  1681.0  1682.0  1681.0  1681.0   1308.160850
2026-11-01 00:02:00+09:00  1681.0  1682.0  1681.0  1682.0  12294.379176
2026-11-01 00:03:00+09:00  1682.0  1682.0  1681.0  1681.0   2314.271095
2026-11-01 00:04:00+09:00  1681.0  1682.0  1681.0  1681.0  17162.427640
                             open    high     low   close        volume
datetime                                                               
2026-11-08 14:55:00+09:00  1586.0  1587.0  1585.0  1586.0  10284.320015
2026-11-08 14:56:00+09:00  1586.0  1586.0  1585.0  1585.0  13918.509497
2026-11-08 14:57:00+09:00  1585.0  1586.0  1585.0  1586.0  39

In [4]:
df_upb_2xrp = df_upb_1xrp

In [ ]:
# 02시부터 07시까지 데이터가 비어있음!!
df_xrp_upb = fetch_korea_month_ohlcv(exchange='upbit', 
                                     symbol='XRP/KRW', 
                                     timeframe='1m',
                                     start_str='2026-10-19 02:00:00',
                                     end_str='2026-10-19 07:59:59')

print(f"Fetched {len(df_xrp_upb)} rows of data for XRP/KRW from Upbit.")
print(df_xrp_upb.head())
print(df_xrp_upb.tail())


[XRP/KRW] 분봉 데이터 수집 시작 from 2026-10-19 02:00:00 to 2026-10-19 07:59:59...
Fetched 57 rows of data for XRP/KRW from Upbit.
                             open    high     low   close         volume
datetime                                                                
2026-10-19 02:00:00+09:00  1721.0  1721.0  1721.0  1721.0    2769.896344
2026-10-19 07:04:00+09:00  1722.0  1748.0  1722.0  1744.0  602520.230465
2026-10-19 07:05:00+09:00  1745.0  1749.0  1741.0  1746.0  256258.504050
2026-10-19 07:06:00+09:00  1745.0  1747.0  1744.0  1744.0   69839.754352
2026-10-19 07:07:00+09:00  1745.0  1745.0  1742.0  1745.0  179442.992992
                             open    high     low   close         volume
datetime                                                                
2026-10-19 07:55:00+09:00  1759.0  1759.0  1758.0  1759.0   12454.610261
2026-10-19 07:56:00+09:00  1759.0  1759.0  1758.0  1758.0   24824.500214
2026-10-19 07:57:00+09:00  1758.0  1759.0  1758.0  1758.0  135244.429225
20

In [ ]:
# [1] vbt 라이브러리를 사용하여 이동평균 교차 전략을 구현하고 백테스트를 수행합니다. by Gemini
vbt_data = vbt.Data.from_data({'XRP/KRW': df_xrp_upb}, download_kwargs={})

price = vbt_data.get('close')
fast_ma = vbt.MA.run(price, window=10)
slow_ma = vbt.MA.run(price, window=30)

entries = fast_ma.ma_crossed_above(slow_ma)
exits = fast_ma.ma_crossed_below(slow_ma)
pf_1 = vbt.Portfolio.from_signals(
    close=price, 
    entries=entries, 
    exits=exits, 
    init_cash=100000, 
    fees=0.0005)
print(pf_1.stats())

Start                         2026-09-30 15:00:00+00:00
End                           2026-10-31 14:59:00+00:00
Period                                            44336
Start Value                                    100000.0
End Value                                  30537.959615
Total Return [%]                              -69.46204
Benchmark Return [%]                         -17.468106
Max Gross Exposure [%]                            100.0
Total Fees Paid                            52876.745894
Max Drawdown [%]                              69.643417
Max Drawdown Duration                           44251.0
Total Trades                                        899
Total Closed Trades                                 898
Total Open Trades                                     1
Open Trade PnL                                -15.26898
Win Rate [%]                                  18.374165
Best Trade [%]                                 2.961926
Worst Trade [%]                               -1

In [ ]:
pf_1.plot().show()

In [5]:
# [2] ta 라이브러리를 사용하여 이동평균과 RSI 지표를 계산하고, 매수/매도 신호를 생성합니다. by Udemy
df_upb_1xrp['ma_short'] = ta.trend.SMAIndicator(close=df_upb_1xrp['close'], window=9).sma_indicator()
df_upb_1xrp['ma_long'] = ta.trend.SMAIndicator(close=df_upb_1xrp['close'], window=21).sma_indicator()

condition_sma = [ df_upb_1xrp['ma_short'] > df_upb_1xrp['ma_long'], df_upb_1xrp['ma_short'] < df_upb_1xrp['ma_long'] ]
df_upb_1xrp['signal_ma'] = np.select(condition_sma, [1, -1], default=0)

df_upb_1xrp['rsi'] = ta.momentum.RSIIndicator(close=df_upb_1xrp['close'], window=7).rsi()

#condition_rsi = [ df_upb_1xrp['rsi'].between(0, 30), df_upb_1xrp['rsi'].between(70, 100) ]
condition_rsi = [ (df_upb_1xrp['rsi'].shift(1) <= 30) & (df_upb_1xrp['rsi'] > 30), (df_upb_1xrp['rsi'].shift(1) >= 70) & (df_upb_1xrp['rsi'] < 70) ]
df_upb_1xrp['signal_rsi'] = np.select(condition_rsi, [1, -1], default=0)


cond_go = [(df_upb_1xrp['signal_ma'] == 1) & (df_upb_1xrp['signal_rsi'] == 1), (df_upb_1xrp['signal_ma'] == -1) & (df_upb_1xrp['signal_rsi'] == -1)]
df_upb_1xrp['go'] = np.select(cond_go, [1, -1], 0)
df_upb_1xrp['go'].value_counts()

go
 0    10929
-1       34
 1       17
Name: count, dtype: int64

In [ ]:
# 돌파 확인 코드 예시 - 샘플 데이터 생성
df_ex = pd.DataFrame({'value': [40, 45, 55, 60, 48, 42, 52]})
threshold = 50

# 1. 상향 돌파 (Cross-up): 직전은 50 이하, 현재는 50 초과
df_ex['cross_up'] = (df_ex['value'].shift(1) <= threshold) & (df_ex['value'] > threshold)

# 2. 하향 붕괴 (Cross-down): 직전은 50 이상, 현재는 50 미만
df_ex['cross_down'] = (df_ex['value'].shift(1) >= threshold) & (df_ex['value'] < threshold)

print(df_ex)


   value  cross_up  cross_down
0     40     False       False
1     45     False       False
2     55      True       False
3     60     False       False
4     48     False        True
5     42     False       False
6     52      True       False


In [6]:
pf_2 = vbt.Portfolio.from_signals(
    close=df_upb_1xrp['close'],
    entries=df_upb_1xrp['go'] == 1,
    exits=df_upb_1xrp['go'] == -1,
    init_cash=100000,
    tp_stop = 0.02,
    sl_stop = 0.005,
    # short_entries = (df_upb_1xrp['go'] == -1), # 공매도 진입?
    # short_exits = (df_upb_1xrp['go'] == 1),  # 공매도 청산?
    fees=0.0005
)

pf_2.stats()

Start                         2026-11-01 00:00:00+09:00
End                           2026-11-08 14:59:00+09:00
Period                                  7 days 15:00:00
Start Value                                    100000.0
End Value                                  97030.859887
Total Return [%]                               -2.96914
Benchmark Return [%]                          -5.651398
Max Gross Exposure [%]                            100.0
Total Fees Paid                             1565.581871
Max Drawdown [%]                               5.688344
Max Drawdown Duration                   5 days 09:35:00
Total Trades                                         16
Total Closed Trades                                  16
Total Open Trades                                     0
Open Trade PnL                                      0.0
Win Rate [%]                                      31.25
Best Trade [%]                                 1.933312
Worst Trade [%]                               -0

In [8]:
pf_2.plot().show()

In [9]:
pf_2.trades.records_readable

,Exit Trade Id,Column,Size,Entry Timestamp,Avg Entry Price,Entry Fees,Exit Timestamp,Avg Exit Price,Exit Fees,PnL,Return,Direction,Status,Position Id
0,0,0,59.212100,2026-11-01 10:46:00+09:00,1688.0,49.975012,2026-11-01 15:30:00+09:00,1687.0,49.945406,-159.132519,-0.001592,Long,Closed,0
1,1,0,59.434766,2026-11-02 00:18:00+09:00,1679.0,49.895486,2026-11-02 04:10:00+09:00,1670.0,49.628030,-634.436409,-0.006358,Long,Closed,1
2,2,0,58.986825,2026-11-02 16:11:00+09:00,1681.0,49.578426,2026-11-02 16:22:00+09:00,1682.0,49.607920,-40.199521,-0.000405,Long,Closed,2
3,3,0,59.068339,2026-11-02 19:57:00+09:00,1678.0,49.558337,2026-11-02 22:04:00+09:00,1682.0,49.676473,137.038547,0.001383,Long,Closed,3
4,4,0,59.291304,2026-11-03 04:08:00+09:00,1674.0,49.626822,2026-11-03 06:34:00+09:00,1686.0,49.982569,611.886260,0.006165,Long,Closed,4
5,5,0,59.267195,2026-11-03 08:44:00+09:00,1685.0,49.932612,2026-11-03 12:05:00+09:00,1676.0,49.665909,-633.003277,-0.006339,Long,Closed,5
6,6,0,57.526108,2026-11-03 22:26:00+09:00,1725.0,49.616268,2026-11-04 00:06:00+09:00,1716.0,49.357401,-616.708644,-0.006215,Long,Closed,6
7,7,0,58.491184,2026-11-04 11:13:00+09:00,1686.0,49.308068,2026-11-04 12:07:00+09:00,1677.0,49.044858,-624.773583,-0.006335,Long,Closed,7
8,8,0,55.708741,2026-11-05 02:33:00+09:00,1759.0,48.995838,2026-11-05 04:01:00+09:00,1749.0,48.717294,-654.800539,-0.006682,Long,Closed,8
9,9,0,55.336670,2026-11-05 08:31:00+09:00,1759.0,48.668601,2026-11-05 10:14:00+09:00,1750.0,48.419586,-595.118213,-0.006114,Long,Closed,9
